In [ ]:
# ============================================================
# CELL 1 — Environment Setup (Gemini Edition — google.colab.ai)
# ============================================================

!pip install -q gradio chromadb PyPDF2 python-docx duckduckgo-search beautifulsoup4 fpdf2 sentence-transformers plotly

import gradio as gr
import chromadb
from chromadb.config import Settings
from google.colab import drive
import google.colab.ai as ai
import os
import torch
import json
import re
import sqlite3
import time
import requests
import csv
from datetime import datetime
from PyPDF2 import PdfReader
import docx
from duckduckgo_search import DDGS
from bs4 import BeautifulSoup
from fpdf import FPDF
import plotly.express as px
import plotly.graph_objects as go

print("✅ Environment ready.")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

# List available models
print("\nAvailable models:")
available_models = ai.list_models()
for model in available_models:
    print(f"  - {model}")

print(f"\n✅ google.colab.ai ready with {len(available_models)} model(s)")




In [ ]:
# ============================================================
# CELL 2 — Core LLM + Sequential Five-Lens Kernel
# ============================================================

available_models = ai.list_models()
MODEL_NAME = available_models[0] if available_models else "google/gemini-2.5-flash"

CERT_STANCES = {
    "Security+": (
        "You are a CompTIA Security+ coach. Think like a broad defensive cybersecurity professional. "
        "Prefer control selection, CIA triad, threat/mitigation pairing, and exam-useful distinctions. "
        "First question in this stance: what control, and why here?"
    ),
    "CEH": (
        "You are a Certified Ethical Hacker coach. Think like an ethical hacker operating only within "
        "authorized boundaries. Teach methodology, scoping, recon-to-report thinking, and defense-informed "
        "offense. Never provide exploit payloads, malware, or instructions to attack real systems."
    ),
    "PMP": (
        "You are a PMP coach. Think like a project leader responsible for delivering outcomes. "
        "Frame answers around stakeholders, constraints (scope/time/cost/quality), risk, hybrid/agile "
        "judgment, and organizational context."
    ),
    "CISA": (
        "You are a CISA coach. Think like an IT auditor examining controls and evidence. "
        "Ask: what is the control, what is the evidence, what is the opinion? Distinguish design vs "
        "operating effectiveness."
    ),
    "CISM": (
        "You are a CISM coach. Think like a security manager responsible for the organizational program. "
        "Ask: who owns this, what is the business impact, what do we fund? Governance over gadgets."
    ),
    "CRISC": (
        "You are a CRISC coach. Think like an IT risk professional. "
        "Ask: what is the risk statement, likelihood/impact, treatment, residual risk, and KRIs?"
    ),
}

FIVE_LENSES = [
    (
        "ANALOGICAL",
        "Compare this to similar professional situations, adjacent certifications, workplace practice, "
        "or known patterns. What is this question like? Draw useful parallels a learner can remember. "
        "Do not give the final exam-style answer yet.",
    ),
    (
        "INDUCTIVE",
        "From the retrieved sources, official-objective language, and examples, what patterns and general "
        "principles emerge? Infer rules the learner can reuse. Do not give the final answer yet.",
    ),
    (
        "CRITICAL",
        "Stress-test the reasoning. Name gaps, common misconceptions, exam distractors (original, not leaked), "
        "thin or unofficial sources, and what a skeptic would argue. Flag if retrieved context is weak. "
        "Do not give the final answer yet.",
    ),
    (
        "RESOLUTION",
        "Reconcile Analogical, Inductive, and Critical into one coherent view. If this competency transfers "
        "from another certification, teach ONLY the certification-specific difference. Still withhold the "
        "short final answer.",
    ),
    (
        "FINAL ANSWER",
        "Give a clear, direct, well-reasoned answer grounded in the four lenses above. State the cognitive "
        "altitude (Know / Understand / Apply / Analyze / Decide). This is exam PREP, not live-exam help. "
        "Do not claim access to real exam items.",
    ),
]

INTEGRITY_REFUSAL = (
    "I cannot help with live examinations, leaked or brain-dump questions, proxy testing, or taking an exam "
    "for you.\n\nThis is a preparation system. I will help you learn objectives, practice with original "
    "scenarios, and judge readiness — not cheat.\n\nIf you are studying, rephrase as a concept, scenario, "
    "or official-objective question (for example: “Explain residual risk for CRISC” or “How would Security+ "
    "expect me to choose a control here?”)."
)


def generate_text(prompt, max_tokens=2048, temperature=0.7, stream=False):
    """Generate text using google.colab.ai."""
    try:
        full_response = []
        for chunk in ai.generate_text(prompt=prompt, model_name=MODEL_NAME, stream=True):
            if chunk is not None:
                full_response.append(chunk)
        return "".join(full_response)
    except Exception as e:
        return f"⚠️ API Error: {str(e)}"


def ask_raw(prompt, max_tokens=2048):
    return generate_text(prompt, max_tokens=max_tokens, temperature=0.1, stream=False)


def safe_ask_raw(prompt, max_tokens=2048):
    try:
        result = ask_raw(prompt, max_tokens=max_tokens)
        if not result or not result.strip():
            return '{"error": "Empty response from LLM. Please try again."}'
        if hasattr(result, "__iter__") and not isinstance(result, (str, dict, list)):
            result = "".join(list(result))
        if not isinstance(result, str):
            result = str(result)
        return result.strip()
    except Exception as e:
        return f'{{"error": "safe_ask_raw failed: {str(e)}"}}'


def check_exam_integrity(text):
    """Return a refusal string if the query looks like cheating / live-exam help, else None."""
    if not text:
        return None
    q = text.lower()
    live = [
        r"\b(pearson\s*vue|prometric|test(ing)?\s+cent(er|re)|exam\s+in\s+progress|live\s+exam)\b",
        r"\bi('m| am) (currently )?(taking|sitting|in) (the |my )?(certification )?(exam|test)\b",
        r"\bhelp me (pass|during) (the |my )?(exam|test)\b",
        r"\btake the (exam|test) for me\b",
        r"\b(proxy\s+test|exam\s+proxy)\b",
    ]
    dumps = [
        r"\b(brain\s*dumps?|exam\s+dumps?|leaked\s+questions?|real\s+exam\s+questions?|actual\s+exam\s+questions?)\b",
        r"\bpaste(d)? (the )?(real |actual )?(exam|dump)\b",
        r"\bguaranteed pass\b",
    ]
    for pat in live + dumps:
        if re.search(pat, q):
            return INTEGRITY_REFUSAL
    return None


def _lens_prompt(question, stance, context, provenance_note, lens_name, lens_instruction, prior):
    stance_text = CERT_STANCES.get(stance, CERT_STANCES["Security+"])
    ctx = context.strip() if context else "(No retrieved knowledge-base context.)"
    prior_block = prior.strip() if prior else "(No prior lenses yet.)"
    prov = provenance_note.strip() if provenance_note else "(No provenance tags.)"
    return f"""You are the 4CBON2 Certification Coach. One intelligence, six packs.
Active stance: {stance}
{stance_text}

This is legitimate exam PREPARATION. Never claim real/leaked exam content. Never give exploit payloads
or attack instructions against real systems. Never assist a live exam.

QUESTION:
{question}

RETRIEVED KNOWLEDGE BASE (may be incomplete; treat unofficial sources as supporting only):
{ctx}

PROVENANCE:
{prov}

PRIOR LENSES:
{prior_block}

NOW WRITE ONLY THIS SECTION:
## {lens_name}
{lens_instruction}

Use the header exactly as shown. Be concrete and professional. 120–220 words unless FINAL ANSWER (then 80–160 words).
"""


def run_five_lens_sequential(question, context=None, stance="Security+", provenance_note=""):
    """Run Analogical → Inductive → Critical → Resolution → Final Answer as sequential calls."""
    refusal = check_exam_integrity(question)
    if refusal:
        return refusal, {"blocked": True}

    prior = ""
    sections = {}
    for name, instruction in FIVE_LENSES:
        prompt = _lens_prompt(
            question=question,
            stance=stance,
            context=context or "",
            provenance_note=provenance_note,
            lens_name=name,
            lens_instruction=instruction,
            prior=prior,
        )
        tokens = 700 if name != "FINAL ANSWER" else 600
        text = generate_text(prompt, max_tokens=tokens, temperature=0.5, stream=False)
        if not text or text.startswith("⚠️") or text.startswith('{"error"'):
            # Fall back to a single-pass five-lens if a sequential call fails.
            return _run_five_lens_single(question, context, stance, provenance_note), {"fallback": True, "failed_lens": name}
        if not text.strip().startswith("##"):
            text = f"## {name}\n{text.strip()}"
        sections[name] = text.strip()
        prior += "\n\n" + text.strip()

    assembled = "\n\n".join(sections[n] for n, _ in FIVE_LENSES)
    header = f"**Stance:** {stance}  \n**Reasoning:** 4CBON2 five-lens (sequential)\n\n"
    return header + assembled, {"blocked": False, "fallback": False, "sections": list(sections.keys())}


def _run_five_lens_single(question, context, stance, provenance_note):
    stance_text = CERT_STANCES.get(stance, CERT_STANCES["Security+"])
    ctx = f"RETRIEVED KNOWLEDGE BASE:\n{context}\n\n" if context else ""
    prov = f"PROVENANCE:\n{provenance_note}\n\n" if provenance_note else ""
    prompt = f"""You are the 4CBON2 Certification Coach.
Active stance: {stance}
{stance_text}

Legitimate exam PREPARATION only. No leaked items, no live-exam help, no exploit payloads.

{ctx}{prov}QUESTION: {question}

Answer using these five sequential lenses, each with a markdown header:
1. ANALOGICAL — comparisons and parallels
2. INDUCTIVE — patterns and principles from evidence
3. CRITICAL — gaps, misconceptions, thin sources
4. RESOLUTION — synthesis; certification-specific difference if transferring
5. FINAL ANSWER — direct answer with cognitive altitude
"""
    return generate_text(prompt, max_tokens=2048, temperature=0.5, stream=False)


def ask_stream(question, context=None, stance="Security+", provenance_note=""):
    """Yield a five-lens answer (sequential kernel)."""
    answer, meta = run_five_lens_sequential(
        question, context=context, stance=stance, provenance_note=provenance_note
    )
    if isinstance(answer, str) and answer.startswith("⚠️"):
        yield answer
        return
    words = answer.split()
    chunk = ""
    for i, word in enumerate(words):
        chunk += word + " "
        if (i + 1) % 8 == 0 or i == len(words) - 1:
            yield chunk
            chunk = ""


def ask(question, context=None, stance="Security+", provenance_note=""):
    full_text = ""
    for chunk in ask_stream(question, context=context, stance=stance, provenance_note=provenance_note):
        full_text += chunk
    return full_text


print(f"✅ Cell 2 ready. Model: {MODEL_NAME}")
print("Five-lens kernel: Analogical → Inductive → Critical → Resolution → Final Answer")
print("Stances:", ", ".join(CERT_STANCES.keys()))


In [ ]:
# ============================================================
# CELL 3 — Drive Mount + ChromaDB (Certification Knowledge Pack)
# ============================================================

drive.mount("/content/drive")
drive_path = "/content/drive/MyDrive/chroma_db_gemini"
os.makedirs(drive_path, exist_ok=True)

client = chromadb.PersistentClient(
    path=drive_path,
    settings=Settings(allow_reset=True),
)

COLLECTION_NAME = "certification_knowledge"

# Public-objective educational seeds (NOT exam items). Provenance is tagged.
CURATED_PACK = [
    {
        "id": "secplus_cia",
        "text": (
            "Security+ objective cluster — Confidentiality, Integrity, Availability (CIA). "
            "Confidentiality protects data from unauthorized disclosure (encryption, access control). "
            "Integrity protects data from unauthorized modification (hashing, signing). "
            "Availability ensures authorized access when needed (redundancy, backups, DDoS defense). "
            "Professional decision: pick the CIA component that the scenario actually threatens, not the one you memorized last."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "cia_triad"},
    },
    {
        "id": "secplus_aaa",
        "text": (
            "Security+ — Authentication, Authorization, Accounting (AAA). Authentication proves identity "
            "(something you know/have/are, MFA). Authorization decides what an authenticated identity may do "
            "(RBAC, least privilege). Accounting/auditing records what happened (logs). "
            "Common confusion: MFA strengthens authentication, not authorization."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "aaa_identity"},
    },
    {
        "id": "secplus_controls",
        "text": (
            "Security+ — Control types and control functions. Managerial (policies, risk assessments), "
            "operational (awareness, configuration processes), technical (firewalls, EDR, encryption). "
            "Preventive, detective, corrective, deterrent, compensating. "
            "Exam-useful judgment: match the control to the residual risk, not to a brand name."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "control_selection"},
    },
    {
        "id": "secplus_threats",
        "text": (
            "Security+ — Threats, vulnerabilities, and mitigations. A threat is a potential cause of harm; "
            "a vulnerability is a weakness that can be exploited; risk is the intersection with impact. "
            "Social engineering, malware classes, supply-chain issues, and unpatched systems are common clusters. "
            "Mitigation is a control that reduces likelihood or impact. Do not confuse a threat actor with a vulnerability."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "threat_vuln_risk"},
    },
    {
        "id": "secplus_ir",
        "text": (
            "Security+ — Incident response phases (preparation, detection/analysis, containment, eradication, "
            "recovery, lessons learned). First priority is often containment of impact while preserving evidence. "
            "Documentation and chain of custody matter. This competency transfers to CISM (program/incident leadership) "
            "and CISA (evidence), with different artifacts."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "incident_response"},
    },
    {
        "id": "secplus_crypto",
        "text": (
            "Security+ — Cryptography in use, not puzzle-solving. Symmetric (one key, speed) vs asymmetric "
            "(key pair, exchange/signatures). Hashing for integrity, not confidentiality. TLS protects data in transit; "
            "full-disk encryption protects data at rest. Certificates bind keys to identities via a trust chain. "
            "Never invent 'exam-only' cipher trivia that is not an objective."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "crypto_use"},
    },
    {
        "id": "secplus_network",
        "text": (
            "Security+ — Secure network architecture. Segmentation, DMZ, zero trust principles (never trust, always verify), "
            "secure protocols vs insecure counterparts, firewalls as policy enforcement, IDS/IPS as detection/prevention. "
            "Architecture decisions are about reducing attack surface and blast radius."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "secure_architecture"},
    },
    {
        "id": "secplus_identity",
        "text": (
            "Security+ — Identity and access management. Least privilege, separation of duties, just-in-time access, "
            "federation/SSO, privileged access management. Account types (user, shared, service, guest) have different risk. "
            "Provisioning and deprovisioning failures are a leading cause of leftover access."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "iam"},
    },
    {
        "id": "secplus_governance",
        "text": (
            "Security+ — Security program management and oversight at a baseline level: policies, standards, procedures, "
            "guidelines; risk management process; third-party risk; privacy overlap; awareness training. "
            "This is the overlap zone with CISM/CRISC/CISA — Security+ wants recognition; those certs want ownership and evidence."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "Security+", "trust": "official_objective", "competency": "governance_baseline"},
    },
    {
        "id": "shared_risk",
        "text": (
            "Shared competency — Risk management (Security+, CRISC, CISM, CISA, PMP). Identify, analyze, treat, monitor. "
            "Risk appetite vs tolerance. Treatment: avoid, mitigate/reduce, transfer, accept. Residual risk remains after treatment. "
            "Artifacts differ: Security+ scenario control choice; CRISC risk register and KRIs; CISM appetite and funding; "
            "CISA whether the process is designed and operating; PMP project RAID/risk log."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "shared", "trust": "public_standard", "competency": "risk_management"},
    },
    {
        "id": "ceh_scope",
        "text": (
            "CEH overlay — Authorized testing only. An ethical hacker operates under written rules of engagement: "
            "scope, time window, allowed techniques, data handling, and reporting. Reconnaissance and enumeration are "
            "methods to understand attack surface so it can be fixed. This system does not provide exploit code or "
            "attack guidance against systems you do not own or have permission to test."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "CEH", "trust": "official_objective", "competency": "authorized_testing"},
    },
    {
        "id": "cisa_evidence",
        "text": (
            "CISA overlay — Evidence and assurance. Auditors evaluate whether controls are suitably designed and "
            "operating effectively. Evidence must be sufficient, reliable, and relevant. Findings should be based on "
            "criteria vs condition, with cause and effect. Security+ names the control; CISA tests it."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "CISA", "trust": "official_objective", "competency": "audit_evidence"},
    },
    {
        "id": "cism_program",
        "text": (
            "CISM overlay — Information security governance and program management. The manager aligns security to "
            "business objectives, assigns ownership, funds the program, and reports risk to leadership. "
            "Incident leadership is about command, communication, and recovery objectives — not packet-level tactics."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "CISM", "trust": "official_objective", "competency": "security_program"},
    },
    {
        "id": "crisc_register",
        "text": (
            "CRISC overlay — IT risk to enterprise risk. A good risk statement names the asset, the threat, the "
            "vulnerability, and the business consequence. Analyze likelihood and impact, choose treatment, record "
            "residual risk, and monitor with KRIs. Controls exist to change risk, not as an end in themselves."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "CRISC", "trust": "official_objective", "competency": "it_risk"},
    },
    {
        "id": "pmp_risk_stakeholder",
        "text": (
            "PMP overlay — Project risk and stakeholders. Project risk threatens objectives (scope, schedule, cost, "
            "quality, benefits). Stakeholder engagement and communication are how decisions land. Hybrid/agile vs "
            "predictive is a judgment about uncertainty and change rate, not a religion. Security work is often delivered as a project."
        ),
        "meta": {"source": "curated", "type": "reference", "pack": "PMP", "trust": "official_objective", "competency": "project_risk_stakeholders"},
    },
]


def _get_or_create_cert_collection():
    try:
        col = client.get_collection(name=COLLECTION_NAME)
        if col.count() == 0:
            raise RuntimeError("empty")
        print(f"✅ Loaded existing collection '{COLLECTION_NAME}' with {col.count()} documents.")
        return col
    except Exception:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
        col = client.create_collection(name=COLLECTION_NAME)
        col.add(
            documents=[d["text"] for d in CURATED_PACK],
            ids=[d["id"] for d in CURATED_PACK],
            metadatas=[d["meta"] for d in CURATED_PACK],
        )
        print(f"✅ Created collection '{COLLECTION_NAME}' with {col.count()} provenance-tagged documents.")
        return col


collection = _get_or_create_cert_collection()
print("Collections available:", [c.name for c in client.list_collections()])
print("Ask a Question database ready (ChromaDB). Answering uses five-lens, not a chatbot dump.")


In [ ]:
# ============================================================
# CELL 4 — 12 Agent Profiles + Tool Registry + DB Helpers
# ============================================================

AGENT_PROFILES = {
    "Default General Assistant": {
        "system_prompt": "You are a helpful general assistant operating within the 4CBON2 architecture.",
        "required_api": None
    },
    "New Autonomous Agent": {
        "system_prompt": """You are the Autonomous Orchestrator Agent for the 4CBON2 ecosystem.
Your role is to:
1. Receive a complex goal from the user.
2. Break it down into 2-4 concrete subtasks.
3. For each subtask, select the most appropriate specialist agent from the list below.
4. Delegate the subtask to that specialist and collect their response.
5. Synthesise all specialist responses into a final, cohesive answer.

Available specialist agents and their expertise:
- Sales Qualification: Lead scoring, BANT criteria, pipeline readiness.
- Legal Document Intelligence: Clause analysis, regulatory compliance, liability extraction.
- Competitive Intelligence: Competitor tracking, market shifts, positioning analysis.
- Customer Engagement: Messaging, sentiment parsing, communication routing.
- Content Strategy: Editorial calendars, copy structuring, keyword architecture.
- Marketing Automation: Campaign triggers, conversion funnels, broadcast sequencing.
- Evidence Management: Data cross-referencing, source auditing, factual verification.
- Scheduling: Time-block coordination, calendar management, bottleneck resolution.
- Legal Intake: Client screening, conflict checks, disclosure structuring.
- Scientific Research: Literature synthesis, data parsing, hypothesis evaluation.
""",
        "required_api": None
    },
    "Sales Qualification": {
        "system_prompt": "You are a Sales Qualification agent. Focus on lead scoring, BANT criteria assessment, and pipeline readiness tracking.",
        "required_api": "CRM_API_KEY"
    },
    "Legal Document Intelligence": {
        "system_prompt": "You are a Legal Document Intelligence agent. Analyze clauses, verify regulatory compliance, and extract liability terms from legal documents.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Competitive Intelligence": {
        "system_prompt": "You are a Competitive Intelligence agent. Scrape competitor updates, track market shifts, and analyze positioning strategies.",
        "required_api": "SEO_API_KEY"
    },
    "Customer Engagement": {
        "system_prompt": "You are a Customer Engagement agent. Craft personalized messaging, parse inbound sentiment, and handle communications routing.",
        "required_api": "COMM_API_KEY"
    },
    "Content Strategy": {
        "system_prompt": "You are a Content Strategy agent. Optimize editorial calendars, structure high-converting copy, and manage keyword architecture.",
        "required_api": "SEO_API_KEY"
    },
    "Marketing Automation": {
        "system_prompt": "You are a Marketing Automation agent. Orchestrate campaign triggers, analyze conversion funnels, and manage broadcast sequences.",
        "required_api": "SOCIAL_SCRAPER_API_KEY"
    },
    "Evidence Management": {
        "system_prompt": "You are an Evidence Management agent. Cross-reference empirical data, audit source trails, and verify factual consistency.",
        "required_api": "S3_VAULT_KEY"
    },
    "Scheduling": {
        "system_prompt": "You are a Scheduling agent. Coordinate time-blocks, handle calendar availability, and resolve logistical bottlenecks.",
        "required_api": "CALENDAR_API_KEY"
    },
    "Legal Intake": {
        "system_prompt": "You are a Legal Intake agent. Screen new client cases, check for conflicts of interest, and structure initial disclosures.",
        "required_api": "DOCUSIGN_API_KEY"
    },
    "Scientific Research": {
        "system_prompt": "You are a Scientific Research agent. Synthesize peer-reviewed literature, parse clinical or technical data, and evaluate hypotheses.",
        "required_api": "PUBMED_API_KEY"
    }
}

LOG_DIR = "/content/drive/MyDrive/4cbon2_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"tool_log_{datetime.now().strftime('%Y%m%d')}.jsonl")

def log_tool_call(tool_name, input_data, result):
    try:
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps({
                "timestamp": datetime.now().isoformat(),
                "tool": tool_name,
                "input": str(input_data)[:500],
                "result_preview": str(result)[:500]
            }) + "\n")
    except Exception as e:
        print(f"⚠️ Log warning: {e}")

STOPWORDS = {
    "of", "the", "a", "an", "for", "to", "in", "on", "is", "are", "and", "or",
    "competitors", "competitor", "alternatives", "alternative", "best", "app",
    "apps", "software", "productivity", "who", "what", "current", "list",
    "similar", "tools", "top", "rated", "reviews", "review", "latest", "new"
}

def _is_relevant(query, text):
    words = re.findall(r"\w+", query)
    keywords = [w for w in words if w.lower() not in STOPWORDS and not (w.isdigit() and len(w) < 4)]
    if not keywords:
        return True
    text_lower = text.lower()
    for kw in keywords:
        if kw.lower() in text_lower:
            return True
    return False

def web_search(query):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return "No results found."
        formatted = []
        for r in results:
            title = r.get("title", "")
            body = r.get("body", "")
            href = r.get("href", "")
            formatted.append(f"{title}\n{body}\n{href}")
        combined = "\n\n".join(formatted)
        return combined if _is_relevant(query, combined) else "Results found but not highly relevant."
    except Exception as e:
        return f"Search error: {e}"

def read_file(file_path):
    try:
        if file_path.endswith(".txt"):
            with open(file_path, "r", errors="ignore") as f:
                return f.read()
        elif file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            texts = []
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    texts.append(t)
            return "\n".join(texts)
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            return "\n".join([para.text for para in doc.paragraphs])
        else:
            return "Unsupported file type. Use .txt, .pdf, or .docx"
    except Exception as e:
        return f"File read error: {e}"

def query_database(sql, db_path="/content/drive/MyDrive/4cbon2_data.db"):
    try:
        cleaned = sql.strip().upper()
        if not cleaned.startswith("SELECT"):
            return "❌ Only SELECT queries are allowed for safety."
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        conn.close()
        return "\n".join([str(row) for row in rows]) if rows else "No results."
    except Exception as e:
        return f"Database error: {e}"

def save_note(content, filename=None):
    try:
        if filename is None:
            filename = f"note_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        path = f"/content/drive/MyDrive/4cbon2_notes/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return f"Note saved to {path}"
    except Exception as e:
        return f"Save error: {e}"

def get_datetime():
    return datetime.now().strftime("Date: %Y-%m-%d | Time: %H:%M:%S")

def http_request(input_str):
    try:
        parsed = json.loads(input_str)
        url = parsed.get("url")
        if not url:
            return "Missing 'url' in input."
        fields = parsed.get("fields", [])
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        content_type = response.headers.get("Content-Type", "")
        if "application/json" in content_type:
            data = response.json()
        else:
            data = response.text
        if isinstance(data, dict) and len(json.dumps(data)) > 4000 and not fields:
            menu = {k: type(v).__name__ for k, v in data.items()}
            return f"Large response. Top-level keys: {json.dumps(menu, indent=2)}"
        if fields:
            result = {}
            for field in fields:
                parts = field.split(".")
                val = data
                for part in parts:
                    if isinstance(val, dict) and part in val:
                        val = val[part]
                    else:
                        val = None
                        break
                result[field] = val
            return json.dumps(result, indent=2)
        return json.dumps(data, indent=2)[:3000] if isinstance(data, (dict, list)) else str(data)[:3000]
    except Exception as e:
        return f"HTTP error: {e}"

def read_csv(file_path):
    try:
        with open(file_path, "r", newline="", errors="ignore") as f:
            rows = list(csv.reader(f))
        if not rows:
            return "CSV is empty."
        header = rows[0]
        preview = rows[1:6]
        return f"Columns: {', '.join(header)}\nRows: {len(rows)-1}\nPreview:\n" + "\n".join([str(r) for r in preview])
    except Exception as e:
        return f"CSV error: {e}"

def write_csv(data_json):
    try:
        rows = json.loads(data_json)
        if not isinstance(rows, list) or not rows:
            return "Input must be a non-empty list of dicts."
        filename = f"export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        path = f"/content/drive/MyDrive/4cbon2_exports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        keys = list(rows[0].keys())
        with open(path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(rows)
        return f"CSV saved to {path} ({len(rows)} rows)"
    except Exception as e:
        return f"CSV write error: {e}"

def generate_pdf(content):
    try:
        filename = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        path = f"/content/drive/MyDrive/4cbon2_reports/{filename}"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", size=12)
        for line in content.split("\n"):
            pdf.multi_cell(0, 8, line)
        pdf.output(path)
        return f"PDF saved to {path}"
    except Exception as e:
        return f"PDF error: {e}"

def scrape_webpage(url):
    try:
        response = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        return "\n".join(lines)[:3000] if lines else "No readable content."
    except Exception as e:
        return f"Scrape error: {e}"

TOOL_REGISTRY = {
    "web_search": {"function": web_search, "description": "Search the web for current information. Input: search query string.", "input": "query"},
    "read_file": {"function": read_file, "description": "Read contents of a .txt, .pdf, or .docx file. Input: file path string.", "input": "file_path"},
    "query_database": {"function": query_database, "description": "Run a SELECT SQL query against the local SQLite database. Input: SQL string.", "input": "sql"},
    "save_note": {"function": save_note, "description": "Save a text note to Google Drive. Input: content string.", "input": "content"},
    "get_datetime": {"function": get_datetime, "description": "Get the current date and time. No input required.", "input": None},
    "http_request": {"function": http_request, "description": "Fetch data from a URL. Input: JSON string like {'url': '...', 'fields': ['field1']}.", "input": "input_str"},
    "read_csv": {"function": read_csv, "description": "Read a CSV file and return columns, row count, and preview. Input: file path string.", "input": "file_path"},
    "write_csv": {"function": write_csv, "description": "Export data to a CSV file on Drive. Input: JSON list of objects.", "input": "data_json"},
    "generate_pdf": {"function": generate_pdf, "description": "Generate a PDF report from text content and save it to Drive. Input: text content string.", "input": "content"},
    "scrape_webpage": {"function": scrape_webpage, "description": "Fetch a webpage and extract its main readable text. Input: URL string.", "input": "url"}
}

def execute_tool(tool_name, tool_input=None):
    if tool_name not in TOOL_REGISTRY:
        return f"Unknown tool: {tool_name}"
    tool = TOOL_REGISTRY[tool_name]
    try:
        if tool["input"] is None:
            result = tool["function"]()
        else:
            result = tool["function"](tool_input)
    except Exception as e:
        result = f"Tool execution error: {e}"
    log_tool_call(tool_name, tool_input, result)
    return result

AGENT_DB_PATH = "/content/drive/MyDrive/4cbon2_agents.db"
os.makedirs(os.path.dirname(AGENT_DB_PATH), exist_ok=True)

def init_agent_db():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS agents (
            agent_id TEXT PRIMARY KEY,
            system_prompt TEXT,
            conversation_history TEXT,
            tools TEXT,
            created_at TEXT,
            updated_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

def load_agent(agent_id):
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT agent_id, system_prompt, conversation_history, tools FROM agents WHERE agent_id = ?",
        (agent_id,)
    )
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            "agent_id": row[0],
            "system_prompt": row[1],
            "conversation_history": json.loads(row[2]) if row[2] else [],
            "tools": json.loads(row[3]) if row[3] else []
        }
    return None

def save_agent(agent_id, system_prompt, conversation_history, tools=None):
    if tools is None:
        tools = []
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT OR REPLACE INTO agents (agent_id, system_prompt, conversation_history, tools, updated_at)
        VALUES (?, ?, ?, ?, ?)
    ''', (
        agent_id,
        system_prompt,
        json.dumps(conversation_history),
        json.dumps(tools),
        datetime.now().isoformat()
    ))
    conn.commit()
    conn.close()

def update_agent_conversation(agent_id, new_messages):
    agent = load_agent(agent_id)
    if agent is None:
        if agent_id in AGENT_PROFILES:
            agent = {
                "agent_id": agent_id,
                "system_prompt": AGENT_PROFILES[agent_id]["system_prompt"],
                "conversation_history": [],
                "tools": []
            }
        else:
            raise ValueError(f"Agent '{agent_id}' not found")
    agent["conversation_history"].extend(new_messages)
    save_agent(agent["agent_id"], agent["system_prompt"], agent["conversation_history"], agent["tools"])

def clear_agent_history(agent_id):
    agent = load_agent(agent_id)
    if agent:
        save_agent(agent_id, agent["system_prompt"], [], agent["tools"])

def get_all_agents():
    conn = sqlite3.connect(AGENT_DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT agent_id FROM agents")
    rows = cursor.fetchall()
    conn.close()
    return [row[0] for row in rows]

def ensure_agents_loaded():
    init_agent_db()
    for agent_id, profile in AGENT_PROFILES.items():
        if load_agent(agent_id) is None:
            save_agent(agent_id, profile["system_prompt"], [], [])
            print(f"✅ Agent '{agent_id}' created in DB.")

ensure_agents_loaded()

print("👥 12 Agent Profiles + 10 Tools loaded.")
print("Agents:", list(AGENT_PROFILES.keys()))
print("Tools:", list(TOOL_REGISTRY.keys()))

In [ ]:
# ============================================================
# CELL 5 — Streaming Multi-Agent Orchestrator
# ============================================================

import json
import re
import os
from datetime import datetime

AUDIT_LOG_PATH = "/content/drive/MyDrive/4cbon2_audit.jsonl"
os.makedirs(os.path.dirname(AUDIT_LOG_PATH), exist_ok=True)

def log_event(event_type, details):
    try:
        record = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "details": details
        }
        with open(AUDIT_LOG_PATH, "a") as f:
            f.write(json.dumps(record) + "\n")
    except Exception as e:
        print(f"⚠️ Audit log warning: {e}")

TASK_MEMORY_PATH = "/content/drive/MyDrive/4cbon2_task_memory.db"
os.makedirs(os.path.dirname(TASK_MEMORY_PATH), exist_ok=True)

def init_task_memory():
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS task_memory (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            goal TEXT,
            subtasks TEXT,
            final_answer TEXT,
            timestamp TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_task_memory(goal, subtasks, final_answer):
    conn = sqlite3.connect(TASK_MEMORY_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO task_memory (goal, subtasks, final_answer, timestamp) VALUES (?, ?, ?, ?)",
        (goal, subtasks, final_answer, datetime.now().isoformat())
    )
    conn.commit()
    conn.close()

init_task_memory()

def _extract_balanced(text, open_ch, close_ch):
    if not text:
        return None
    start = text.find(open_ch)
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == open_ch:
                depth += 1
            elif ch == close_ch:
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]
    return None

def extract_json_object(text):
    return _extract_balanced(text, '{', '}')

def extract_json_array(text):
    return _extract_balanced(text, '[', ']')

def _parse_agent_json(raw_response):
    if not raw_response:
        return None
    candidate = extract_json_object(raw_response)
    try:
        if candidate:
            return json.loads(candidate)
        return json.loads(raw_response)
    except Exception:
        return None

def build_tool_descriptions():
    lines = []
    for name, info in TOOL_REGISTRY.items():
        input_desc = info["input"] if info["input"] else "None"
        lines.append(f"- **{name}**: {info['description']} (input: {input_desc})")
    return "\n".join(lines)

def execute_agent(agent_id, user_message, context="", max_tool_iterations=3):
    agent = load_agent(agent_id)
    if agent is None:
        return f"❌ Agent '{agent_id}' not found."

    system_prompt = agent["system_prompt"]
    history = agent.get("conversation_history", [])

    tool_instructions = f"""You have access to these tools:
{build_tool_descriptions()}

To use a tool, respond with ONLY this JSON:
{{"action": "tool_call", "tool": "<tool_name>", "tool_input": "<input or null>"}}

To answer directly, respond with ONLY this JSON:
{{"action": "final_answer", "content": "<your answer>"}}

Valid JSON only. Max {max_tool_iterations} tool calls before final_answer."""

    prompt_parts = [f"System: {system_prompt}", tool_instructions]
    if context:
        prompt_parts.append(f"Context from other agents:\n{context}")
    for msg in history[-4:]:
        prompt_parts.append(f"{msg['role']}: {msg['content']}")
    prompt_parts.append(f"User: {user_message}")

    tool_call_log = []
    final_content = None

    for iteration in range(max_tool_iterations):
        full_prompt = "\n\n".join(prompt_parts)
        raw_response = safe_ask_raw(full_prompt, max_tokens=2048)

        parsed = _parse_agent_json(raw_response)

        if parsed is None:
            final_content = raw_response
            break

        action = parsed.get("action")

        if action == "tool_call":
            tool_name = parsed.get("tool", "")
            tool_input = parsed.get("tool_input")

            if tool_name not in TOOL_REGISTRY:
                prompt_parts.append(f"Assistant: {raw_response}")
                prompt_parts.append(f"Tool Result: ❌ Unknown tool '{tool_name}'. Available: {', '.join(TOOL_REGISTRY.keys())}")
                continue

            tool_result = execute_tool(tool_name, tool_input)
            tool_call_log.append({"tool": tool_name, "input": tool_input, "result": str(tool_result)[:300]})
            log_event("agent_tool_call", {
                "agent_id": agent_id,
                "tool": tool_name,
                "input": tool_input,
                "result_preview": str(tool_result)[:200]
            })

            prompt_parts.append(f"Assistant: {raw_response}")
            prompt_parts.append(f"Tool Result ({tool_name}): {str(tool_result)[:2000]}")
            continue

        elif action == "final_answer":
            final_content = parsed.get("content", raw_response)
            break
        else:
            final_content = raw_response
            break

    if final_content is None:
        forced_prompt = "\n\n".join(prompt_parts) + "\n\nYou must respond now with ONLY the final_answer JSON format."
        raw_response = safe_ask_raw(forced_prompt, max_tokens=2048)
        parsed = _parse_agent_json(raw_response)
        final_content = parsed.get("content", raw_response) if parsed else raw_response

    if not final_content or final_content.strip() == "" or "could not generate" in final_content.lower():
        fallback_prompt = f"You are a {agent_id} specialist. Provide a best-practice framework for your domain with key metrics, benchmarks, workflows, data collection methods, and improvement strategies."
        final_content = safe_ask_raw(fallback_prompt, max_tokens=1024)
        if not final_content or final_content.strip() == "":
            final_content = f"⚠️ {agent_id} could not generate a response. Please provide more specific instructions or data."

    if tool_call_log:
        tools_used_note = "\n\n---\n🔧 **Tools used:** " + ", ".join(t["tool"] for t in tool_call_log)
        final_content = final_content + tools_used_note

    update_agent_conversation(agent_id, [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_content}
    ])
    return final_content

def synthesize_batch(batch, goal, batch_num, total_batches):
    prompt = f"""Synthesise part {batch_num} of {total_batches} of a strategic audit.

Goal: {goal}

Specialist reports:
{json.dumps(batch, indent=2)}

Provide a CONCISE summary (under 200 words) of key findings, themes, and gaps."""
    result = safe_ask_raw(prompt, max_tokens=1024)
    print(f"[DEBUG] Batch {batch_num}/{total_batches}: {len(result)} chars")
    return result

def synthesize_final(batch_summaries, goal):
    prompt = f"""Create the final strategic report.

Goal: {goal}

Batch summaries:
{json.dumps(batch_summaries, indent=2)}

Synthesise into a cohesive report with:
1. Executive summary
2. Clear sections
3. Integrated insights
4. Prioritised action plan

Final Report:"""
    print(f"[DEBUG] Final synthesis prompt: {len(prompt)} chars")
    result = safe_ask_raw(prompt, max_tokens=2048)
    print(f"[DEBUG] Final synthesis response: {len(result)} chars")
    return result

def generate_fallback_plan(goal):
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan = []
    keyword_map = {
        "sales": "Sales Qualification", "lead": "Sales Qualification", "pipeline": "Sales Qualification",
        "legal": "Legal Document Intelligence", "contract": "Legal Document Intelligence",
        "compliance": "Legal Document Intelligence", "liability": "Legal Document Intelligence",
        "competitor": "Competitive Intelligence", "market": "Competitive Intelligence",
        "position": "Competitive Intelligence", "customer": "Customer Engagement",
        "engagement": "Customer Engagement", "messaging": "Customer Engagement",
        "sentiment": "Customer Engagement", "content": "Content Strategy",
        "seo": "Content Strategy", "blog": "Content Strategy", "social": "Content Strategy",
        "marketing": "Marketing Automation", "campaign": "Marketing Automation",
        "funnel": "Marketing Automation", "evidence": "Evidence Management",
        "data": "Evidence Management", "fact": "Evidence Management",
        "schedule": "Scheduling", "calendar": "Scheduling", "time": "Scheduling",
        "intake": "Legal Intake", "client": "Legal Intake", "conflict": "Legal Intake",
        "research": "Scientific Research", "paper": "Scientific Research", "technology": "Scientific Research"
    }
    used = set()
    for keyword, specialist in keyword_map.items():
        if keyword in goal.lower() and specialist not in used:
            plan.append({
                "subtask": f"Analyse {keyword} aspects",
                "specialist": specialist,
                "instructions": f"Provide comprehensive analysis related to '{keyword}'."
            })
            used.add(specialist)
    if not plan:
        plan = [
            {"subtask": "Analyse market and competitors", "specialist": "Competitive Intelligence", "instructions": "Provide trends and competitor mapping."},
            {"subtask": "Identify legal risks", "specialist": "Legal Document Intelligence", "instructions": "Summarise key compliance issues."},
            {"subtask": "Recommend strategy", "specialist": "Content Strategy", "instructions": "Develop a strategic plan."}
        ]
    return plan[:12]

def enforce_explicit_specialists(plan, goal):
    goal_lower = goal.lower()
    planned = {item.get("specialist") for item in plan}
    for name in AGENT_PROFILES.keys():
        if name in ("New Autonomous Agent", "Default General Assistant"):
            continue
        if name.lower() in goal_lower and name not in planned:
            plan.append({
                "subtask": f"Explicit request: apply {name} expertise",
                "specialist": name,
                "instructions": f"The user explicitly requested {name} analysis. Address it directly."
            })
    return plan

def run_orchestrator_stream(goal, model_name=None):
    yield f"🚀 **Orchestrator started:** {goal}\n\n---\n"
    log_event("orchestrator_start", {"goal": goal})

    yield "🔄 **Step 1:** Clearing orchestrator history...\n"
    clear_agent_history("New Autonomous Agent")
    yield "✅ Done.\n\n"

    yield "🧠 **Step 2:** Generating plan...\n"
    specialists = [a for a in AGENT_PROFILES.keys() if a not in ["New Autonomous Agent", "Default General Assistant"]]
    plan_prompt = f"""You are the Autonomous Orchestrator Agent.

User goal: {goal}

Break this into up to 12 subtasks using EVERY relevant specialist from:
{', '.join(specialists)}

If the goal explicitly names a specialist, you MUST include it.

Output as JSON array:
[
    {{"subtask": "...", "specialist": "...", "instructions": "..."}},
    ...
]

Valid JSON only. No other text."""
    plan_response = safe_ask_raw(plan_prompt, max_tokens=1024)
    yield f"📝 Plan response: {len(plan_response)} chars\n"

    try:
        candidate = extract_json_array(plan_response)
        if candidate:
            plan = json.loads(candidate)
        else:
            plan = json.loads(plan_response)
        if not isinstance(plan, list) or len(plan) == 0:
            raise ValueError("Empty plan")
    except Exception as e:
        yield f"⚠️ Plan parsing error: {e}\nUsing fallback.\n\n"
        plan = generate_fallback_plan(goal)
        yield f"📋 Fallback plan: {len(plan)} steps.\n\n"

    before = len(plan)
    plan = enforce_explicit_specialists(plan, goal)
    if len(plan) > before:
        yield f"🛡️ Guardrail: added {len(plan) - before} specialist(s).\n\n"

    subtask_results = []
    for i, item in enumerate(plan, 1):
        subtask = item.get("subtask", f"Subtask {i}")
        specialist = item.get("specialist", "Default General Assistant")
        instructions = item.get("instructions", "Analyze thoroughly.")
        yield f"\n---\n**Step {i}/{len(plan)}:** {subtask}\n👤 `{specialist}`\n📋 {instructions}\n\n"

        if load_agent(specialist) is None:
            yield f"⚠️ '{specialist}' not found. Using Default.\n"
            specialist = "Default General Assistant"

        clear_agent_history(specialist)
        context = json.dumps([
            {"step": s["step"], "subtask": s["subtask"], "preview": s["result"][:150] + "..." if len(s["result"]) > 150 else s["result"]}
            for s in subtask_results
        ], indent=2)

        yield f"⏳ Executing `{specialist}`...\n"
        result = execute_agent(
            specialist,
            f"Task: {subtask}\n\nInstructions: {instructions}\n\nContext: {context}"
        )
        subtask_results.append({"step": i, "subtask": subtask, "specialist": specialist, "result": result})
        yield f"✅ `{specialist}` done.\n📄 {result[:300]}{'...' if len(result) > 300 else ''}\n\n"

    yield "\n---\n🧬 **Final Synthesis...**\n"

    if not subtask_results:
        fallback = execute_agent("Default General Assistant", f"Answer directly: {goal}")
        final_answer = f"⚠️ No specialists generated. Fallback:\n\n{fallback}"
    else:
        batch_size = 3
        batches = [subtask_results[i:i+batch_size] for i in range(0, len(subtask_results), batch_size)]
        summaries = []
        for idx, batch in enumerate(batches, 1):
            yield f"📦 Synthesising batch {idx}/{len(batches)}...\n"
            summary = synthesize_batch(batch, goal, idx, len(batches))
            summaries.append({"batch": idx, "specialists": [r["specialist"] for r in batch], "summary": summary})
            yield f"✅ Batch {idx} done.\n\n"

        yield "🧬 Final synthesis...\n"
        final_answer = synthesize_final(summaries, goal)
        if not final_answer or not final_answer.strip():
            final_answer = "⚠️ Synthesis empty. Raw reports:\n\n" + "\n\n".join([s["result"] for s in subtask_results])

    subtasks_summary = "\n".join([f"Step {s['step']}: {s['subtask']} → {s['specialist']}" for s in subtask_results]) if subtask_results else "No subtasks."
    save_task_memory(goal, subtasks_summary, final_answer)

    yield "\n---\n# 🧠 Multi-Agent Report\n\n"
    yield f"## 🎯 Goal\n{goal}\n\n"
    yield f"## 📋 Execution\n{subtasks_summary}\n\n"
    if subtask_results:
        yield "## 📊 Reports\n"
        for s in subtask_results:
            yield f"\n### Step {s['step']}: {s['subtask']} ({s['specialist']})\n{s['result']}\n"
    yield f"\n## 🧬 Final Answer\n{final_answer}\n\n---\n*Generated by 4CBON2 (Gemini Edition)*\n"

    log_event("orchestrator_complete", {"goal": goal, "steps": len(subtask_results)})

def run_orchestrator(goal, model_name=None):
    full = ""
    for chunk in run_orchestrator_stream(goal, model_name):
        full += chunk
    return full

def run_agent(goal, system_override=None):
    return run_orchestrator(goal)

print("⚙️ Orchestrator ready.")
print("Agents:", get_all_agents())

In [ ]:
# ============================================================
# CELL 6 — RAG + Integrity + Security+ Competency Graph v0 + Readiness
# ============================================================

def chunk_text(text, max_chunk_size=800, overlap=100):
    if not text or not text.strip():
        return []
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 30]
    if len(paragraphs) >= 3:
        return paragraphs
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) < max_chunk_size:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    if chunks:
        return chunks
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]


def process_document(file_obj):
    if file_obj is None:
        return "No file uploaded."
    try:
        file_path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)
        text = read_file(file_path)
        if text.startswith(("File read error", "Unsupported file type")):
            return f"❌ {text}"
        if not text or not text.strip():
            return "❌ No extractable text found in file."
        low = text.lower()
        if re.search(r"\b(brain\s*dump|leaked exam|actual exam questions|real exam questions)\b", low):
            return (
                "❌ Upload rejected by exam-integrity policy. "
                "Do not index leaked or purported real examination items. "
                "Upload official objectives, your notes, or openly licensed study material instead."
            )
        chunks = chunk_text(text)
        if not chunks:
            return "❌ Could not create chunks from document."
        base_name = os.path.basename(file_path)
        ids = [f"user_{base_name}_{i}_{int(time.time())}" for i in range(len(chunks))]
        metadatas = [
            {
                "source": "user_upload",
                "type": "uploaded",
                "pack": "user",
                "trust": "user_corpus",
                "filename": base_name,
                "competency": "",
            }
            for _ in chunks
        ]
        collection.add(documents=chunks, ids=ids, metadatas=metadatas)
        return (
            f"✅ Indexed {len(chunks)} chunks from '{base_name}' as user corpus (trust=user_corpus). "
            f"Total KB docs: {collection.count()}"
        )
    except Exception as e:
        return f"❌ Upload error: {e}"


def retrieve_with_provenance(kb_name, question, n_results=5):
    col = client.get_collection(kb_name)
    results = col.query(
        query_texts=[question],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    docs = (results.get("documents") or [[]])[0]
    metas = (results.get("metadatas") or [[]])[0]
    dists = (results.get("distances") or [[]])[0]
    blocks = []
    provenance_lines = []
    for i, doc in enumerate(docs):
        meta = metas[i] if i < len(metas) and metas[i] else {}
        dist = dists[i] if i < len(dists) else None
        source = meta.get("source", "unknown")
        trust = meta.get("trust", "unsourced")
        pack = meta.get("pack", "")
        comp = meta.get("competency", "")
        filename = meta.get("filename", "")
        dist_s = f"{dist:.3f}" if isinstance(dist, (int, float)) else "n/a"
        tag = f"[{i+1}] source={source} trust={trust} pack={pack} competency={comp} dist={dist_s}"
        if filename:
            tag += f" file={filename}"
        provenance_lines.append(tag)
        blocks.append(f"{tag}\n{doc}")
    context = "\n\n".join(blocks) if blocks else ""
    provenance_note = "\n".join(provenance_lines) if provenance_lines else "No chunks retrieved."
    return context, provenance_note, docs, metas


def handle_ask_question(kb_name, question, stance="Security+"):
    """Retrieve from the Ask a Question database, then answer with sequential five-lens."""
    if not question or not question.strip():
        return "Please enter a valid question."
    refusal = check_exam_integrity(question)
    if refusal:
        log_inquiry(question, stance, competencies=[], blocked=True)
        return refusal
    try:
        context, provenance_note, docs, metas = retrieve_with_provenance(kb_name, question)
        comps = match_competencies(question, metas)
        log_inquiry(question, stance, competencies=comps, blocked=False)
        answer, meta = run_five_lens_sequential(
            question,
            context=context,
            stance=stance,
            provenance_note=provenance_note,
        )
        footer = "\n\n---\n**Retrieved provenance**\n" + provenance_note
        if comps:
            footer += "\n\n**Mapped competencies:** " + ", ".join(comps)
        if meta.get("blocked"):
            return answer
        return answer + footer
    except Exception as e:
        return f"Error: {e}"


# ---------- Security+ competency graph v0 (proving ground) ----------

SECPLUS_GRAPH = [
    {
        "id": "cia_triad",
        "domain": "General Security Concepts",
        "name": "Apply CIA triad",
        "altitude": "Apply",
        "tags": ["cia", "confidentiality", "integrity", "availability", "encryption", "hash"],
        "transfers": ["CISM", "CISA", "CRISC"],
    },
    {
        "id": "aaa_identity",
        "domain": "General Security Concepts",
        "name": "Distinguish authentication vs authorization vs accounting",
        "altitude": "Analyze",
        "tags": ["aaa", "authentication", "authorization", "mfa", "accounting"],
        "transfers": ["CISM", "CISA"],
    },
    {
        "id": "control_selection",
        "domain": "General Security Concepts",
        "name": "Select an appropriate control type/function",
        "altitude": "Decide",
        "tags": ["control", "preventive", "detective", "compensating", "managerial", "technical"],
        "transfers": ["CISA", "CISM", "CRISC"],
    },
    {
        "id": "threat_vuln_risk",
        "domain": "Threats, Vulnerabilities, and Mitigations",
        "name": "Separate threat, vulnerability, and risk",
        "altitude": "Analyze",
        "tags": ["threat", "vulnerability", "malware", "phishing", "mitigation"],
        "transfers": ["CEH", "CRISC", "CISM"],
    },
    {
        "id": "incident_response",
        "domain": "Security Operations",
        "name": "Order and justify incident response actions",
        "altitude": "Decide",
        "tags": ["incident", "containment", "eradication", "recovery", "forensics"],
        "transfers": ["CISM", "CISA"],
    },
    {
        "id": "crypto_use",
        "domain": "Security Architecture",
        "name": "Choose crypto for confidentiality vs integrity vs transit vs rest",
        "altitude": "Apply",
        "tags": ["crypto", "tls", "hashing", "symmetric", "asymmetric", "certificate"],
        "transfers": ["CEH"],
    },
    {
        "id": "secure_architecture",
        "domain": "Security Architecture",
        "name": "Reason about segmentation and zero trust",
        "altitude": "Analyze",
        "tags": ["segmentation", "dmz", "zero trust", "firewall", "ids", "ips"],
        "transfers": ["CEH", "CISM"],
    },
    {
        "id": "iam",
        "domain": "Security Operations",
        "name": "Apply least privilege and identity lifecycle",
        "altitude": "Apply",
        "tags": ["iam", "least privilege", "rbac", "sso", "privileged", "deprovision"],
        "transfers": ["CISA", "CISM"],
    },
    {
        "id": "governance_baseline",
        "domain": "Security Program Management and Oversight",
        "name": "Recognize policy vs standard vs procedure and basic risk process",
        "altitude": "Understand",
        "tags": ["policy", "standard", "procedure", "governance", "third-party", "privacy"],
        "transfers": ["CISM", "CISA", "CRISC", "PMP"],
    },
    {
        "id": "risk_management",
        "domain": "Security Program Management and Oversight",
        "name": "Apply risk identification, treatment, and residual risk",
        "altitude": "Decide",
        "tags": ["risk", "residual", "appetite", "tolerate", "transfer", "accept", "kri"],
        "transfers": ["CRISC", "CISM", "CISA", "PMP"],
    },
]

GRAPH_BY_ID = {n["id"]: n for n in SECPLUS_GRAPH}

LEARNER_DB_PATH = "/content/drive/MyDrive/4cbon2_learner.db"
os.makedirs(os.path.dirname(LEARNER_DB_PATH), exist_ok=True)


def init_learner_db():
    conn = sqlite3.connect(LEARNER_DB_PATH)
    c = conn.cursor()
    c.execute(
        """
        CREATE TABLE IF NOT EXISTS evidence (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            pack TEXT,
            competency_id TEXT,
            altitude TEXT,
            score REAL,
            source TEXT,
            note TEXT
        )
        """
    )
    c.execute(
        """
        CREATE TABLE IF NOT EXISTS inquiries (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            stance TEXT,
            question TEXT,
            competencies TEXT,
            blocked INTEGER
        )
        """
    )
    conn.commit()
    conn.close()


init_learner_db()


def log_inquiry(question, stance, competencies, blocked=False):
    try:
        conn = sqlite3.connect(LEARNER_DB_PATH)
        c = conn.cursor()
        c.execute(
            "INSERT INTO inquiries (timestamp, stance, question, competencies, blocked) VALUES (?,?,?,?,?)",
            (
                datetime.now().isoformat(),
                stance,
                question[:2000],
                ",".join(competencies),
                1 if blocked else 0,
            ),
        )
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"⚠️ inquiry log: {e}")


def record_evidence(competency_id, score, altitude="Apply", source="probe", note=""):
    if competency_id not in GRAPH_BY_ID:
        return f"❌ Unknown competency '{competency_id}'."
    try:
        conn = sqlite3.connect(LEARNER_DB_PATH)
        c = conn.cursor()
        c.execute(
            "INSERT INTO evidence (timestamp, pack, competency_id, altitude, score, source, note) VALUES (?,?,?,?,?,?,?)",
            (
                datetime.now().isoformat(),
                "Security+",
                competency_id,
                altitude,
                float(score),
                source,
                note[:500],
            ),
        )
        conn.commit()
        conn.close()
        return f"✅ Recorded {competency_id} score={score:.2f} ({source})."
    except Exception as e:
        return f"❌ {e}"


def match_competencies(question, metas=None):
    q = (question or "").lower()
    hits = []
    for node in SECPLUS_GRAPH:
        if any(tag in q for tag in node["tags"]) or node["id"] in q or node["name"].lower() in q:
            hits.append(node["id"])
    if metas:
        for m in metas:
            if not m:
                continue
            cid = m.get("competency") or ""
            if cid in GRAPH_BY_ID and cid not in hits:
                hits.append(cid)
    return hits


def competency_states():
    conn = sqlite3.connect(LEARNER_DB_PATH)
    c = conn.cursor()
    c.execute(
        "SELECT competency_id, altitude, score, source, timestamp FROM evidence WHERE pack='Security+' ORDER BY timestamp"
    )
    rows = c.fetchall()
    conn.close()
    state = {
        n["id"]: {
            "node": n,
            "scores": [],
            "last": None,
            "sources": set(),
            "max_altitude": "None",
        }
        for n in SECPLUS_GRAPH
    }
    alt_rank = {"None": 0, "Know": 1, "Understand": 2, "Apply": 3, "Analyze": 4, "Decide": 5}
    for cid, altitude, score, source, ts in rows:
        if cid not in state:
            continue
        state[cid]["scores"].append(float(score))
        state[cid]["last"] = (ts, float(score), source)
        state[cid]["sources"].add(source)
        if alt_rank.get(altitude, 0) > alt_rank.get(state[cid]["max_altitude"], 0):
            state[cid]["max_altitude"] = altitude
    return state


def readiness_score_security_plus():
    """Certification Readiness Score v0 — Security+ proving ground. Not a pass guarantee."""
    state = competency_states()
    total = len(SECPLUS_GRAPH)
    evidenced = 0
    mastery_vals = []
    domain_scores = {}
    for cid, st in state.items():
        domain = st["node"]["domain"]
        domain_scores.setdefault(domain, [])
        if st["scores"]:
            evidenced += 1
            # recent-weighted mean of last 3
            recent = st["scores"][-3:]
            m = sum(recent) / len(recent)
            mastery_vals.append(m)
            domain_scores[domain].append(m)
        else:
            domain_scores[domain].append(0.0)

    coverage = evidenced / total if total else 0.0
    mastery = (sum(mastery_vals) / len(mastery_vals)) if mastery_vals else 0.0

    # Retention: last evidence within 14 days and score >= 0.7
    now = datetime.now()
    retained = 0
    for st in state.values():
        if not st["last"]:
            continue
        ts, score, _ = st["last"]
        try:
            dt = datetime.fromisoformat(ts)
        except Exception:
            continue
        if (now - dt).days <= 14 and score >= 0.7:
            retained += 1
    retention = retained / total if total else 0.0

    # Mock fidelity: evidence source == mock
    mock_scores = []
    for st in state.values():
        if "mock" in st["sources"] and st["scores"]:
            mock_scores.append(st["scores"][-1])
    mock = (sum(mock_scores) / len(mock_scores)) if mock_scores else None

    # Weak-domain penalty: any domain mean < 0.4 after it has evidence or after 3+ nodes evidenced overall
    weak = []
    for domain, vals in domain_scores.items():
        mean = sum(vals) / len(vals)
        if mean < 0.4:
            weak.append((domain, mean))

    # Consistency: inverse of variance of last scores
    last_scores = [st["scores"][-1] for st in state.values() if st["scores"]]
    if len(last_scores) >= 3:
        mu = sum(last_scores) / len(last_scores)
        var = sum((x - mu) ** 2 for x in last_scores) / len(last_scores)
        consistency = max(0.0, 1.0 - var)
    else:
        consistency = 0.5 if last_scores else 0.0

    # Integrity of evidence: down-weight if only inquiries, no probes
    probe_count = 0
    for st in state.values():
        if st["sources"] & {"probe", "mock", "scenario"}:
            probe_count += 1
    integrity = min(1.0, 0.4 + 0.6 * (probe_count / max(1, evidenced))) if evidenced else 0.3

    mock_component = mock if mock is not None else mastery * 0.5
    raw = (
        0.18 * coverage
        + 0.28 * mastery
        + 0.14 * retention
        + 0.18 * mock_component
        + 0.12 * consistency
        + 0.10 * integrity
    )
    score = round(100 * raw)
    if weak and evidenced >= 3:
        score = min(score, 68)  # cap when a domain is red

    blockers = []
    for cid, st in state.items():
        recent = st["scores"][-3:] if st["scores"] else []
        mean = (sum(recent) / len(recent)) if recent else 0.0
        if mean < 0.6:
            blockers.append((st["node"]["name"], st["node"]["domain"], mean, st["node"]["altitude"]))
    blockers.sort(key=lambda x: x[2])

    explanation = [
        f"**Security+ Readiness Score (v0): {score}/100**",
        "",
        "_This is evidence of preparation, not a pass guarantee. Original items only — no real exam content._",
        "",
        f"- Coverage (nodes with evidence): {coverage:.0%} ({evidenced}/{total})",
        f"- Mastery depth (recent probe mean): {mastery:.0%}",
        f"- Retention (strong evidence ≤14 days): {retention:.0%}",
        f"- Mock fidelity: {'n/a (no mocks yet)' if mock is None else f'{mock:.0%}'}",
        f"- Consistency: {consistency:.0%}",
        f"- Evidence integrity (probes vs curiosity): {integrity:.0%}",
    ]
    if weak:
        explanation.append("- Weak domains: " + ", ".join(f"{d} ({m:.0%})" for d, m in weak))
    if blockers:
        explanation.append("")
        explanation.append("**Highest-leverage gaps**")
        for name, domain, mean, alt in blockers[:5]:
            explanation.append(f"- {name} [{domain}] — {mean:.0%} (target altitude: {alt})")
    else:
        explanation.append("")
        explanation.append("No major gaps recorded yet — run a pulse diagnose.")
    return score, "\n".join(explanation), state


def render_graph_markdown():
    score, expl, state = readiness_score_security_plus()
    lines = [expl, "", "### Competency graph (Security+ proving ground)", ""]
    current_domain = None
    for node in SECPLUS_GRAPH:
        if node["domain"] != current_domain:
            current_domain = node["domain"]
            lines.append(f"**{current_domain}**")
        st = state[node["id"]]
        recent = st["scores"][-3:] if st["scores"] else []
        mean = (sum(recent) / len(recent)) if recent else None
        mark = "·" if mean is None else ("✓" if mean >= 0.75 else ("~" if mean >= 0.5 else "✗"))
        mean_s = "—" if mean is None else f"{mean:.0%}"
        xfer = ", ".join(node["transfers"]) if node["transfers"] else "—"
        lines.append(
            f"- [{mark}] `{node['id']}` {node['name']} — {mean_s} — altitude {node['altitude']} — transfers: {xfer}"
        )
    return "\n".join(lines)


# Original pulse items (NOT real exam questions). Each maps to a competency.
PULSE_BANK = [
    {
        "competency": "cia_triad",
        "altitude": "Apply",
        "q": "A hospital needs to stop unauthorized disclosure of patient records on lost laptops. Which CIA component is the primary concern, and which control class best addresses it?",
        "options": [
            "A) Availability — redundant servers",
            "B) Confidentiality — full-disk encryption and access control",
            "C) Integrity — file hashing on the EHR",
            "D) Availability — DDoS mitigation",
        ],
        "answer": "B",
        "why": "Lost laptops threaten disclosure (confidentiality). Encryption and access control reduce unauthorized disclosure. Hashing is integrity; redundancy/DDoS are availability.",
    },
    {
        "competency": "aaa_identity",
        "altitude": "Analyze",
        "q": "A user authenticates with MFA but can still open payroll files they should not see. What failed?",
        "options": [
            "A) Authentication — MFA was insufficient",
            "B) Authorization — permissions are too broad",
            "C) Accounting — logs are missing",
            "D) Identification — the username is wrong",
        ],
        "answer": "B",
        "why": "Identity was proven (MFA). The failure is what they are allowed to do — authorization / least privilege.",
    },
    {
        "competency": "control_selection",
        "altitude": "Decide",
        "q": "After a phishing click, leadership wants a control that STOPS similar emails from reaching inboxes. Which function is that?",
        "options": [
            "A) Detective — SIEM alert after delivery",
            "B) Corrective — wipe the endpoint after malware runs",
            "C) Preventive — email filtering / secure email gateway",
            "D) Compensating — extra insurance policy",
        ],
        "answer": "C",
        "why": "Stopping delivery is preventive. Alerts are detective; wiping after the fact is corrective; insurance transfers residual impact.",
    },
    {
        "competency": "incident_response",
        "altitude": "Decide",
        "q": "Ransomware is encrypting a file server now. Evidence may be needed later. What is the best immediate professional move?",
        "options": [
            "A) Reimage every workstation immediately",
            "B) Pay the ransom so recovery is faster",
            "C) Contain the spread (isolate the server/segment) while preserving volatile evidence where feasible",
            "D) Skip to lessons learned and write the report",
        ],
        "answer": "C",
        "why": "Contain impact first without blindly destroying evidence. Payment is a business/legal decision, not the IR first move. Lessons learned come last.",
    },
    {
        "competency": "risk_management",
        "altitude": "Decide",
        "q": "A risk is treated with a compensating control. Residual risk is still above appetite. What should happen next?",
        "options": [
            "A) Declare the risk closed because a control exists",
            "B) Record residual risk and escalate / choose further treatment or formal acceptance by the owner",
            "C) Ignore it until the annual audit",
            "D) Convert it to a vulnerability scan finding only",
        ],
        "answer": "B",
        "why": "Controls do not automatically close risk. Residual risk must be owned — further treat or accept against appetite.",
    },
]


def pulse_prompt_text():
    lines = ["### Security+ pulse diagnose (original items — not real exam content)", ""]
    for i, item in enumerate(PULSE_BANK, 1):
        lines.append(f"**{i}. [{item['competency']}]** {item['q']}")
        lines.extend(item["options"])
        lines.append("")
    lines.append("Enter answers as five letters, e.g. `B B C C B`")
    return "\n".join(lines)


def score_pulse(answer_line):
    if not answer_line or not str(answer_line).strip():
        return "❌ Enter five letters (A–D).", render_graph_markdown()
    letters = re.findall(r"[A-Da-d]", str(answer_line))
    if len(letters) < len(PULSE_BANK):
        return f"❌ Need {len(PULSE_BANK)} answers; got {len(letters)}.", render_graph_markdown()
    lines = ["### Pulse results", ""]
    correct = 0
    for item, given in zip(PULSE_BANK, letters):
        g = given.upper()
        ok = g == item["answer"]
        correct += int(ok)
        record_evidence(
            item["competency"],
            1.0 if ok else 0.0,
            altitude=item["altitude"],
            source="probe",
            note=f"pulse {item['competency']} {g}",
        )
        mark = "✅" if ok else "❌"
        lines.append(f"{mark} `{item['competency']}` you={g} correct={item['answer']}")
        lines.append(item["why"])
        lines.append("")
    lines.append(f"**Pulse score: {correct}/{len(PULSE_BANK)}**")
    lines.append("")
    lines.append(render_graph_markdown())
    return "\n".join(lines), render_graph_markdown()


def record_manual_mastery(competency_id, score_pct, note):
    try:
        s = float(score_pct) / 100.0
    except Exception:
        return "❌ Score must be 0–100.", render_graph_markdown()
    s = min(1.0, max(0.0, s))
    msg = record_evidence(competency_id, s, altitude=GRAPH_BY_ID.get(competency_id, {}).get("altitude", "Apply"), source="scenario", note=note or "")
    return msg, render_graph_markdown()


# ============================================================
# DATA DASHBOARD FUNCTIONS (unchanged role: task memory viz)
# ============================================================

def load_task_memory_data():
    try:
        conn = sqlite3.connect(TASK_MEMORY_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT goal, subtasks, final_answer, timestamp FROM task_memory ORDER BY timestamp DESC LIMIT 20")
        rows = cursor.fetchall()
        conn.close()
        if not rows:
            return None, "No task memory data found. Run some agent tasks first!"
        data = []
        for row in rows:
            goal, subtasks, final_answer, timestamp = row
            data.append({
                "goal": goal,
                "subtasks": subtasks,
                "final_answer": final_answer[:200] + "..." if len(final_answer) > 200 else final_answer,
                "timestamp": timestamp,
                "subtask_count": len(subtasks.split("\n")) if subtasks else 0,
                "answer_length": len(final_answer) if final_answer else 0,
            })
        return data, None
    except Exception as e:
        return None, f"Error loading task memory: {str(e)}"


def create_plotly_dashboard():
    data, error = load_task_memory_data()
    if error:
        return None, error
    if not data:
        return None, "No data available"
    figures = []
    timestamps = [d["timestamp"] for d in data]
    goals = [d["goal"][:50] + "..." if len(d["goal"]) > 50 else d["goal"] for d in data]
    answer_lengths = [d["answer_length"] for d in data]
    fig1 = go.Figure(data=[go.Bar(x=timestamps, y=answer_lengths, text=goals, textposition="auto", marker_color="rgb(55, 83, 109)")])
    fig1.update_layout(title="Task Response Length Over Time", xaxis_title="Timestamp", yaxis_title="Response Length (characters)", height=400)
    figures.append(fig1)
    subtask_counts = [d["subtask_count"] for d in data]
    fig2 = go.Figure(data=[go.Histogram(x=subtask_counts, nbinsx=10, marker_color="rgb(26, 118, 255)")])
    fig2.update_layout(title="Distribution of Subtasks per Task", xaxis_title="Number of Subtasks", yaxis_title="Frequency", height=400)
    figures.append(fig2)
    from collections import Counter
    all_words = []
    stop_words = {"the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "from", "as", "is", "was", "are", "were", "been", "be", "have", "has", "had", "do", "does", "did", "will", "would", "could", "should", "may", "might", "must", "can"}
    for d in data:
        words = d["goal"].lower().split()
        all_words.extend([w for w in words if w not in stop_words and len(w) > 3])
    word_counts = Counter(all_words).most_common(15)
    if word_counts:
        fig3 = go.Figure(data=[go.Bar(x=[wc[0] for wc in word_counts], y=[wc[1] for wc in word_counts], marker_color="rgb(255, 127, 14)")])
        fig3.update_layout(title="Most Common Words in Task Goals", xaxis_title="Word", yaxis_title="Frequency", height=400)
        figures.append(fig3)
    return figures, None


print("✅ RAG + five-lens ask path + Security+ graph v0 + readiness ready.")


In [ ]:
# ============================================================
# CELL 7 — Master UI (Certification Coach — no Builder)
# ============================================================
import gradio as gr
import traceback

try:
    gr.close_all()
except Exception:
    pass


def run_agent(goal, use_gemini_only, enable_additional, *api_keys):
    """Run the orchestrator with optional API key injection controlled by checkboxes."""
    yield f"✅ Using Gemini ({MODEL_NAME}) — authenticated with OAuth 2.0.\n\n"

    if use_gemini_only:
        yield "🔒 **Gemini Only mode** — all additional API keys ignored.\n\n"
    elif enable_additional:
        yield "🔓 **Additional APIs enabled** — injecting optional API keys.\n\n"
        key_names = [
            "CALENDAR_API_KEY", "CRM_API_KEY", "COMM_API_KEY", "VISION_API_KEY",
            "DOCUSIGN_API_KEY", "SOCIAL_SCRAPER_API_KEY", "SEO_API_KEY", "S3_VAULT_KEY", "PUBMED_API_KEY",
        ]
        for name, val in zip(key_names, api_keys):
            if val and val.strip():
                os.environ[name] = val.strip()
    else:
        yield "🔒 **Additional APIs disabled** — using Gemini only.\n\n"

    try:
        for chunk in run_orchestrator_stream(goal):
            yield chunk
    except Exception as e:
        yield f"❌ Orchestrator error: {str(e)}"
    finally:
        key_names = [
            "CALENDAR_API_KEY", "CRM_API_KEY", "COMM_API_KEY", "VISION_API_KEY",
            "DOCUSIGN_API_KEY", "SOCIAL_SCRAPER_API_KEY", "SEO_API_KEY", "S3_VAULT_KEY", "PUBMED_API_KEY",
        ]
        for name in key_names:
            os.environ.pop(name, None)


def ask_five_lens(question, stance):
    if not question or not question.strip():
        return "❌ Enter a question.", "❌ No question"
    try:
        stance = stance or "Security+"
        answer = handle_ask_question(COLLECTION_NAME, question, stance=stance)
        blocked = "cannot help with live examinations" in answer.lower()
        status = "🛡️ Integrity refusal" if blocked else ("✅ Five-lens complete" if not str(answer).startswith("❌") else str(answer)[:80])
        return answer, status
    except Exception as e:
        return f"❌ Error: {str(e)}\n{traceback.format_exc()}", "❌ Failed"


with gr.Blocks(title="4CBON2 — Certification Coach") as demo:
    gr.Markdown(
        """
# 4CBON2 — Unified Certification Coach
One intelligence · six packs · **five-lens** answering (Analogical → Inductive → Critical → Resolution → Final Answer)

Prep system, not a cheating system. No leaked items. No Builder tab.
"""
    )

    with gr.Tabs():
        with gr.TabItem("❓ Ask a Question"):
            gr.Markdown(
                """
Search the **knowledge-base database** (curated packs + your uploads), then answer through the **sequential five-lens** kernel.

Pick a professional stance. Overlapping competencies transfer; the Resolution lens should teach only the certification-specific difference.
"""
            )
            with gr.Row():
                stance_dd = gr.Dropdown(
                    choices=list(CERT_STANCES.keys()),
                    value="Security+",
                    label="Certification stance",
                )
            question_box = gr.Textbox(
                label="Your question",
                lines=3,
                placeholder="Example: How do I choose between hashing and encryption?  /  What is residual risk?",
            )
            ask_btn = gr.Button("Ask with five-lens", variant="primary")
            ask_status = gr.Textbox(label="Status", interactive=False)
            ask_output = gr.Markdown(label="Five-lens answer")
            ask_btn.click(
                fn=ask_five_lens,
                inputs=[question_box, stance_dd],
                outputs=[ask_output, ask_status],
            )

        with gr.TabItem("📁 Upload Documents"):
            gr.Markdown(
                """
Add material to the **same Ask a Question database**. Files are tagged `trust=user_corpus` and are never treated as official exam content.

Leaked / purported real exam items are rejected.
"""
            )
            file_input = gr.File(label="Upload file", file_types=[".txt", ".pdf", ".docx"])
            upload_output = gr.Textbox(label="Status", interactive=False)
            upload_btn = gr.Button("Process & Index", variant="primary")
            upload_btn.click(fn=process_document, inputs=[file_input], outputs=[upload_output])

        with gr.TabItem("📈 Security+ Readiness"):
            gr.Markdown(
                """
## Proving ground — Security+ competency graph v0

Mastery is evidence, not video completion. Pulse items are **original**. Readiness Score is not a pass guarantee.
"""
            )
            graph_md = gr.Markdown(value=render_graph_markdown())
            refresh_graph_btn = gr.Button("Refresh graph & score")
            refresh_graph_btn.click(fn=render_graph_markdown, outputs=[graph_md])

            gr.Markdown("### Pulse diagnose")
            pulse_box = gr.Markdown(value=pulse_prompt_text())
            pulse_answers = gr.Textbox(label="Answers (five letters)", placeholder="B B C C B")
            pulse_btn = gr.Button("Score pulse", variant="primary")
            pulse_out = gr.Markdown()
            pulse_btn.click(fn=score_pulse, inputs=[pulse_answers], outputs=[pulse_out, graph_md])

            gr.Markdown("### Manual evidence (scenario / lab you completed)")
            with gr.Row():
                man_comp = gr.Dropdown(choices=[n["id"] for n in SECPLUS_GRAPH], label="Competency")
                man_score = gr.Slider(0, 100, value=70, step=5, label="Score %")
            man_note = gr.Textbox(label="Note", placeholder="e.g. tabletop: ransomware containment")
            man_btn = gr.Button("Record evidence")
            man_status = gr.Textbox(label="Status", interactive=False)
            man_btn.click(
                fn=record_manual_mastery,
                inputs=[man_comp, man_score, man_note],
                outputs=[man_status, graph_md],
            )

        with gr.TabItem("🤖 Agent Mode"):
            gr.Markdown("Optional multi-agent orchestration (parked from 4CBON2). Not required for certification coaching.")
            with gr.Row():
                with gr.Column(scale=2):
                    profile_selector = gr.Dropdown(
                        choices=list(AGENT_PROFILES.keys()),
                        value="New Autonomous Agent",
                        label="Agent Profile",
                    )
                    agent_goal = gr.Textbox(
                        label="Goal / Instructions",
                        lines=3,
                        placeholder="e.g. Draft a Security+ study week focused on incident response...",
                    )
                    chk_gemini_only = gr.Checkbox(
                        label="Use Only Gemini API",
                        value=True,
                        info="When checked, only Gemini is used. All other API keys are ignored.",
                    )
                    chk_enable_additional = gr.Checkbox(
                        label="Enable Additional APIs",
                        value=False,
                        info="When checked (and Gemini-only is OFF), optional API key fields become available.",
                    )
                    agent_btn = gr.Button("Run Orchestrator", variant="primary")
                with gr.Column(scale=1):
                    additional_keys_accordion = gr.Accordion("🔑 Optional API Keys", open=False, visible=False)
                    with additional_keys_accordion:
                        t_cal = gr.Textbox(label="Calendar", type="password")
                        t_crm = gr.Textbox(label="CRM", type="password")
                        t_comm = gr.Textbox(label="Comm", type="password")
                        t_vision = gr.Textbox(label="Vision/OCR", type="password")
                        t_ds = gr.Textbox(label="DocuSign", type="password")
                        t_social = gr.Textbox(label="Social", type="password")
                        t_seo = gr.Textbox(label="SEO", type="password")
                        t_s3 = gr.Textbox(label="S3/Vault", type="password")
                        t_pubmed = gr.Textbox(label="PubMed", type="password")

            def update_keys_visibility(gemini_only, enable_additional):
                show = (not gemini_only) and enable_additional
                return gr.Accordion(visible=show, open=show)

            chk_gemini_only.change(
                fn=update_keys_visibility,
                inputs=[chk_gemini_only, chk_enable_additional],
                outputs=[additional_keys_accordion],
            )
            chk_enable_additional.change(
                fn=update_keys_visibility,
                inputs=[chk_gemini_only, chk_enable_additional],
                outputs=[additional_keys_accordion],
            )

            agent_output = gr.Textbox(label="Execution Log & Output", lines=25, interactive=False)
            agent_btn.click(
                fn=run_agent,
                inputs=[
                    agent_goal,
                    chk_gemini_only, chk_enable_additional,
                    t_cal, t_crm, t_comm, t_vision, t_ds, t_social, t_seo, t_s3, t_pubmed,
                ],
                outputs=agent_output,
            )

        with gr.TabItem("📊 Data Dashboard"):
            gr.Markdown("Task memory visualization for Agent Mode (not certification readiness).")
            dashboard_btn = gr.Button("🔄 Load Dashboard", variant="primary")
            dashboard_output = gr.Textbox(label="Status", interactive=False)
            with gr.Row():
                dashboard_plot1 = gr.Plot(label="Task Timeline")
                dashboard_plot2 = gr.Plot(label="Subtask Distribution")
            with gr.Row():
                dashboard_plot3 = gr.Plot(label="Goal Word Frequency")

            def load_dashboard():
                try:
                    figures, error = create_plotly_dashboard()
                    if error:
                        return f"❌ {error}", None, None, None
                    if not figures:
                        return "❌ No figures generated", None, None, None
                    fig1 = figures[0] if len(figures) > 0 else None
                    fig2 = figures[1] if len(figures) > 1 else None
                    fig3 = figures[2] if len(figures) > 2 else None
                    return f"✅ Loaded {len(figures)} visualization(s)", fig1, fig2, fig3
                except Exception as e:
                    return f"❌ Error: {str(e)}", None, None, None

            dashboard_btn.click(
                fn=load_dashboard,
                inputs=[],
                outputs=[dashboard_output, dashboard_plot1, dashboard_plot2, dashboard_plot3],
            )

        with gr.TabItem("📊 Agent Status"):
            gr.Markdown("View agent conversation histories.")
            refresh_btn = gr.Button("Refresh")
            agent_status_display = gr.Markdown("Click refresh to load.")

            def get_agent_status():
                output = "## Agent Status\n\n"
                for agent_id in get_all_agents():
                    agent = load_agent(agent_id)
                    history_len = len(agent.get("conversation_history", []))
                    output += f"- **{agent_id}**: {history_len} messages\n"
                return output

            refresh_btn.click(fn=get_agent_status, outputs=[agent_status_display])
            demo.load(fn=get_agent_status, outputs=[agent_status_display])

demo.queue()
demo.launch(inline=False, share=True)


In [ ]:
# ============================================================
# CELL 8 — Download This Notebook
# ============================================================
from google.colab import files
files.download('4CBOn2_Cyber_Cert_Coach.ipynb')
